[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/odd_elasticity/odd_elasticity.ipynb)

# Odd Elasticity

This example computes the cyclic mechanical work and overdamped lattice waves of a planar active solid whose bonds push sideways when stretched, following [Scheibner and collaborators, *Odd Elasticity*](https://arxiv.org/abs/1902.07760). Passive springs pull solely along their axes and return every bit of work stored in a closed deformation loop, whereas internally powered bonds add a transverse force in an actuation plane that breaks reciprocity without breaking translational or rotational invariance. A triangular lattice of such bonds exchanges net work around closed shear cycles and supports propagating waves even when viscous drag overwhelms inertia on every particle.

Displacements are modelled as intrinsic antivectors and forces as extrinsic vectors paired by mechanical work `force & displacement`, so a bond with longitudinal unit force `unit_force = direction.dual_inverse()` turns its active push inside the bivector `actuation_plane` as `unit_force | actuation_plane`. Because the resulting response `push * (unit_force & Displacement)` has an antisymmetric part under `&`, a closed displacement loop encloses bivector area from `start ^ end` and exchanges net work proportional to `actuation_plane.scalar_product(enclosed_area)`. Pulling forces back through `motion.adjugate()(response(motion))` preserves that work pairing from a single bond to macroscopic cell shears and across the lattice coboundary, where metric mobility `Force.dual() / drags` converts nodal forces into overdamped velocities.

In Cartesian tensor-index notation, a bond along $\hat n_i = (\cos\theta, \sin\theta)$ writes its force law with the two-dimensional Levi-Civita symbol $\varepsilon_{ij}$ as $f_i = (-k\,\hat n_i - k^o \varepsilon_{ik}\hat n_k)\,\hat n_j \Delta u_j$, where mirror reflections require an extra $\det R$ sign convention on $\varepsilon_{ik}$ to separate handed actuation from vector components. Coarse-graining to the rank-four modulus tensor $\sigma_{ij} = K_{ijkl}\partial_l u_k$ expands $K_{ijkl}$ in $2 \times 2$ basis matrices with $K_{ijkl} \neq K_{klij}$ to isolate the odd shear coupling $K^o$, and overdamped lattice motion sums bond matrices componentwise as $\gamma\,\dot u^{(a)}_i = \sum_{b \sim a} k^{(ab)}_{ij}(u^{(b)}_j - u^{(a)}_j)$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.mechanics.odd_elasticity import render, scenarios
from examples.util.animation import show_states
from numga import NumpyContext, stack
from numga.algebras import VGA2D
from numga.sparse import Sparse, SparseExtensor

np.set_printoptions(precision=4, suppress=True)

# Euclidean algebra of the plane, x+y+.
ga = VGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
# Displacements and shear amplitudes are intrinsic antivectors; forces are extrinsic vectors paired by work.
Displacement = ga.gatype.antivector()                                        # Antivector-
Force = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Response = ga.gatype((Force, Displacement))                                  # Force+ <- Displacement-
Shear = ga.gatype.antivector()                                               # Antivector-

# Unit displacements dual to the coordinate force planes.
x, y = stack((mv.x, mv.y)).dual()                                            # [] Displacement-, Displacement-

## Work from an active bond

A passive spring answers longitudinal extension with a purely longitudinal force, making its response symmetric under the work pairing so every closed excursion returns zero net work. Adding a transverse push `unit_force | actuation_plane` gives the response an antisymmetric part whose work around a closed endpoint loop equals the activity paired with the oriented area swept by the endpoint, reversing sign when the loop is traversed in reverse.

In [ ]:
stiffness = 1
activity = 3
rest_length = 1
parallel_radius = 0.25
transverse_radius = 0.18
n_steps = 126
duration_ms = 40
force_scale = 0.55
phases = np.linspace(0, 2 * np.pi, n_steps + 1)
direction = x                                                                # Displacement-
actuation_plane = mv.xy                                                      # Bivector+

In [ ]:
def bond_response(direction: Displacement, actuation_plane: Bivector, stiffness: float, activity: float) -> Response:
    """Each stretched bond answers with a longitudinal restoring force and a transverse active push (`[...] Force+ <- Displacement-`)."""
    # Unit force along the bond whose pairing reads longitudinal extension.
    unit_force = direction.dual_inverse()                                    # [...] Force+
    push = -stiffness * unit_force + activity * (unit_force | actuation_plane)  # [...] Force+
    return push * (unit_force & Displacement)                                # [...] Force+ <- Displacement-


def cycle_states(response: Response, cycle: Displacement) -> Iterator[tuple[Displacement, Force, Scalar]]:
    """Yields `(displacement, force, work)` along consecutive straight segments of `cycle` (`[...] Displacement-`)."""
    work = mv.scalar([0])                                                    # Scalar+
    yield cycle[0], response(cycle[0]), work
    for start, end in zip(cycle[:-1], cycle[1:]):
        # Midpoint evaluation integrates a linear force response without quadrature error on each segment.
        work = work + (response((start + end) / 2) & (end - start))          # Scalar+
        yield end, response(end), work


anchor = 0 * direction                                                       # Displacement-
reference = rest_length * direction                                          # Displacement-
response = bond_response(direction, actuation_plane, stiffness, activity)    # Force+ <- Displacement-
across = direction | actuation_plane                                         # Displacement-
# Stretch the unit circle independently along and across the bond axis.
ellipse = (
    parallel_radius * direction * (direction.dual_inverse() & Displacement)
    + transverse_radius * across * (across.dual_inverse() & Displacement)
)                                                                            # Displacement- <- Displacement-
loop_turns = (-actuation_plane * (phases / 2)).exp()                         # [n_steps + 1] Rotor+
bond_loop = ellipse(loop_turns >> direction)                                 # [n_steps + 1] Displacement-
bond_stream = (
    (reference + displacement, force, work)
    for displacement, force, work in cycle_states(response, bond_loop)
)
bond_states = show_states(
    bond_stream, render.bond_frames, anchor, reference, reference + bond_loop, force_scale,
    name="odd_elasticity_bond", duration_ms=duration_ms, n_frames=n_steps + 1,
)

As the teal bond's free endpoint traces the grey elliptical path counter-clockwise around its ringed rest position, the orange arrow shows the total bond force tilted sideways from the bond axis, and the peach fan accumulates the swept area proportional to the work delivered.

In [ ]:
bond_work = bond_states[-1][2]                                               # Scalar+
enclosed_area = (bond_loop[:-1] ^ bond_loop[1:]).sum(axis=0) / 2             # Bivector+
area_work = -activity * actuation_plane.scalar_product(enclosed_area)        # Scalar+

print("work around the loop:   ", bond_work.to_array())
print("activity times its area:", area_work.to_array())

## Shear cycle on a lattice patch

Combining active bonds at sixty-degree intervals in a triangular unit cell cancels any preferred bond axis while preserving the handedness of `actuation_plane`. Two area-preserving pure shears move the three bond families linearly, and pulling their bond responses back through `bond_motion.adjugate()` yields an isotropic odd shear response that delivers three bonds' worth of cyclic work per unit cell while loading only the exterior boundary of a finite patch.

In [ ]:
spacing = 1
shear_amplitude = 0.14
n_patch_columns, n_patch_rows = 7, 7
patch_force_scale = 1.3
bond_angles = np.arange(3) * np.pi / 3
patch_topology, patch_coordinates = scenarios.triangular_patch(n_patch_columns, n_patch_rows, spacing)
patch_reference = mv.vector(patch_coordinates).dual().field()                # Displacement-[V]

In [ ]:
def lattice_response(incidence: Sparse, response: Response) -> SparseExtensor:
    """Adjugate pullback across `incidence` (`Scalar+[E <- V]`) maps bond responses (`(Force+ <- Displacement-)[E]`) to nodal forces (`(Force+ <- Displacement-)[V <- V]`)."""
    bond_displacement = incidence * Displacement                             # (Displacement- <- Displacement-)[E <- V]
    return bond_displacement.adjugate()(response(bond_displacement))         # (Force+ <- Displacement-)[V <- V]


# Pure shear along the axes and at forty-five degrees, combined into a bilinear strain map.
axial_shear = x * (mv.x & Displacement) - y * (mv.y & Displacement)          # Displacement- <- Displacement-
diagonal_shear = x * (mv.y & Displacement) + y * (mv.x & Displacement)       # Displacement- <- Displacement-
shear_strain = (mv.x & Shear) * axial_shear + (mv.y & Shear) * diagonal_shear  # Displacement- <- (Shear-, Displacement-)

# One bond from each of the three lattice families in a unit cell.
family_turns = (-actuation_plane * (bond_angles / 2)).exp()                  # [3] Rotor+
bond_directions = family_turns >> x                                          # [3] Displacement-
cell_response = bond_response(bond_directions, actuation_plane, stiffness, activity)  # [3] Force+ <- Displacement-
bond_motion = shear_strain(Shear, spacing * bond_directions)                 # [3] Displacement- <- Shear-
# Pulling bond forces back through the bond motion yields the macroscopic shear response per cell.
shear_response = bond_motion.adjugate()(cell_response(bond_motion)).sum(axis=0)  # Force+ <- Shear-

# Assembling the same bond responses across the finite patch exposes its boundary loads.
edge_directions = (patch_topology.d[0] * patch_reference).normalized()       # Displacement-[E]
edge_response = bond_response(edge_directions, actuation_plane, stiffness, activity)  # (Force+ <- Displacement-)[E]
patch_response = lattice_response(patch_topology.d[0], edge_response)        # (Force+ <- Displacement-)[V <- V]
patch_motion = shear_strain(Shear, patch_reference)                          # (Displacement- <- Shear-)[V]
patch_forces = patch_response(patch_motion)                                  # (Force+ <- Shear-)[V]
shear_loop = shear_amplitude * (loop_turns >> x)                             # [n_steps + 1] Shear-
shear_stream = (
    (patch_reference + patch_motion(shear), patch_forces(shear), work)
    for shear, _, work in cycle_states(shear_response, shear_loop)
)
shear_states = show_states(
    shear_stream, render.patch_frames, patch_reference, patch_topology, patch_force_scale,
    name="odd_elasticity_shear", duration_ms=duration_ms, n_frames=n_steps + 1,
)

Around the closed circle of horizontal and diagonal shears, interior bond forces cancel in pairs at every six-valent vertex while the orange boundary arrows rotate continuously around the perimeter to exchange net work with whatever holds the border.

## Overdamped waves without inertia

In a viscous medium where drag balances elastic force instantaneously on every vertex, metric mobility `Force.dual() / drags` maps nodal forces directly to velocities without inertial coasting. Passive stiffness alone relaxes every perturbation monotonically, whereas transverse bond activity couples longitudinal compression to transverse motion across the lattice coboundary so a localized wavepacket oscillates as it decays.

In [ ]:
n_wave_columns, n_wave_rows = 21, 11
wave_stiffness = 0.2
wave_activity = 4
drag = 1
wave_amplitude = 0.15
wave_width = 2.5
wave_number = 1.2
n_frames = 180
n_steps_per_frame = 2
dt = 1 / 60
displacement_scale = 4
extension_scale = 0.04
tracked_index = (n_wave_rows // 2) * n_wave_columns + n_wave_columns // 2
wave_topology, wave_coordinates = scenarios.triangular_patch(n_wave_columns, n_wave_rows, spacing)
V, E, F = wave_topology.n_elements
wave_reference = mv.vector(wave_coordinates).dual().field()                  # Displacement-[V]

In [ ]:
def velocity_map(incidence: Sparse, response: Response, drags: Scalar) -> SparseExtensor:
    """Metric mobility against `drags` maps nodal bond forces to overdamped velocities (`(Displacement- <- Displacement-)[V <- V]`)."""
    # Viscous drag maps each nodal force to a velocity along its metric dual direction.
    mobility = Force.dual() / drags                                          # (Displacement- <- Force+)[V]
    return mobility(lattice_response(incidence, response))                   # (Displacement- <- Displacement-)[V <- V]


wave_directions = (wave_topology.d[0] * wave_reference).normalized()         # Displacement-[E]
wave_response = bond_response(wave_directions, actuation_plane, wave_stiffness, wave_activity)  # (Force+ <- Displacement-)[E]
drags = mv.scalar([[drag]] * V).field()                                      # Scalar+[V]
velocities = velocity_map(wave_topology.d[0], wave_response, drags)          # (Displacement- <- Displacement-)[V <- V]
envelope = (-wave_reference.scalar_norm_squared() / (2 * wave_width**2)).exp()  # Scalar+[V]
initial_displacement = wave_amplitude * x * envelope * (wave_number * (mv.x & wave_reference)).cos()  # Displacement-[V]

In [ ]:
def rk4_step(velocities: SparseExtensor, displacement: Displacement, dt: float) -> Displacement:
    """Fourth-order Runge-Kutta integration advances `displacement` (`Displacement-[V]`) by one step of duration `dt`."""
    first_rate = velocities(displacement)                                    # Displacement-[V]
    second_rate = velocities(displacement + first_rate * (dt / 2))           # Displacement-[V]
    third_rate = velocities(displacement + second_rate * (dt / 2))          # Displacement-[V]
    fourth_rate = velocities(displacement + third_rate * dt)                 # Displacement-[V]
    return displacement + (first_rate + 2 * second_rate + 2 * third_rate + fourth_rate) * (dt / 6)


def wave_states(
    reference: Displacement, incidence: Sparse, bond_directions: Displacement, velocities: SparseExtensor,
    initial: Displacement, n_frames: int, n_steps_per_frame: int, dt: float,
) -> Iterator[tuple[Displacement, Scalar]]:
    """Yields nodal positions (`Displacement-[V]`) and signed bond extensions (`Scalar+[E]`) over `n_frames` frames of `n_steps_per_frame` steps."""
    displacement = initial                                                   # Displacement-[V]
    for _ in range(n_frames):
        extensions = bond_directions.dual_inverse() & (incidence * displacement)  # Scalar+[E]
        yield reference + displacement, extensions
        for _ in range(n_steps_per_frame):
            displacement = rk4_step(velocities, displacement, dt)            # Displacement-[V]


wave_stream = wave_states(
    wave_reference, wave_topology.d[0], wave_directions, velocities,
    initial_displacement, n_frames, n_steps_per_frame, dt,
)
wave_records = show_states(
    wave_stream, render.wave_frames, wave_reference, wave_topology,
    displacement_scale, extension_scale, tracked_index,
    name="odd_elasticity_wave", duration_ms=duration_ms, n_frames=n_frames,
)

Blue bonds are compressed and orange bonds extended around the pale grey rest lattice, with displacements magnified fourfold while the gold marker traces an elliptical particle orbit driven solely by transverse bond forces against drag.

In [ ]:
# checks
passive_response = bond_response(direction, actuation_plane, stiffness, 0)   # Force+ <- Displacement-
increments = bond_loop[1:] - bond_loop[:-1]                                  # [n_steps] Displacement-
midpoints = (bond_loop[1:] + bond_loop[:-1]) / 2                             # [n_steps] Displacement-
passive_work = (passive_response(midpoints) & increments).sum(axis=0)        # Scalar+
shear_area = (shear_loop[:-1] ^ shear_loop[1:]).sum(axis=0) / 2              # Bivector+
shear_work = -3 * activity * actuation_plane.scalar_product(shear_area)      # Scalar+
interior_forces = stack([force.batch() for _, force, _ in shear_states])[:, patch_topology.interior[0]]  # [n_steps + 1, ...] Force+
initial_centre = (wave_records[0][0] - wave_reference).batch().sum(axis=0)   # Displacement-
final_centre = (wave_records[-1][0] - wave_reference).batch().sum(axis=0)    # Displacement-

assert ((bond_work - area_work).abs() < 1e-12).all()
assert (passive_work.abs() < 1e-12).all()
assert ((shear_states[-1][2] - shear_work).abs() < 1e-12).all()
assert (interior_forces.norm() < 1e-12).all()
assert ((final_centre - initial_centre).norm() < 1e-12).all()
assert (wave_records[-1][1].scalar_norm_squared().batch().sum()
        < wave_records[0][1].scalar_norm_squared().batch().sum()).all()